In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
train_path="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
test_path="/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"

In [8]:

import matplotlib
matplotlib.use("Agg")  # no display needed, just save files
import matplotlib.pyplot as plt

OPTIONS = ["A", "B", "C", "D", "E"]


def basic_overview(train_df: pd.DataFrame, test_df: pd.DataFrame):
    print("=" * 60)
    print("BASIC OVERVIEW")
    print("=" * 60)
    print(f"Train shape: {train_df.shape}   Test shape: {test_df.shape}")
    print(f"\nTrain columns: {list(train_df.columns)}")
    print(f"Test columns:  {list(test_df.columns)}")

    print("\n--- Missing values (train) ---")
    print(train_df.isnull().sum())
    print("\n--- Missing values (test) ---")
    print(test_df.isnull().sum())

    dupes = train_df.duplicated(subset=["prompt"]).sum()
    print(f"\nDuplicate prompts in train: {dupes}")


def answer_distribution(train_df: pd.DataFrame):
    """
    Checks whether correct answers are evenly spread across A-E.
    A skew here is a real signal — if 'C' is correct 40% of the time,
    a model could shortcut by just guessing C first without reading
    the question at all. You want to know this before you're impressed
    by your baseline's score.
    """
    print("\n" + "=" * 60)
    print("ANSWER DISTRIBUTION (checking for positional bias)")
    print("=" * 60)
    counts = train_df["answer"].value_counts().reindex(OPTIONS, fill_value=0)
    pct = (counts / counts.sum() * 100).round(1)
    for opt in OPTIONS:
        print(f"  {opt}: {counts[opt]:4d}  ({pct[opt]}%)")
    expected = 100 / len(OPTIONS)
    max_dev = (pct - expected).abs().max()
    if max_dev > 5:
        print(f"\n  ⚠ Deviation of {max_dev:.1f}pp from uniform ({expected:.1f}%) — "
              f"worth mentioning in your report's error analysis.")
    else:
        print(f"\n  ✓ Roughly uniform (max deviation {max_dev:.1f}pp) — no obvious positional bias.")

    fig, ax = plt.subplots(figsize=(5, 4))
    counts.plot(kind="bar", ax=ax, color="#4C72B0")
    ax.set_title("Correct Answer Distribution")
    ax.set_xlabel("Option")
    ax.set_ylabel("Count")
    fig.tight_layout()

    plt.close(fig)


def text_length_analysis(train_df: pd.DataFrame):
    """
    Word-count distributions for prompts and options.
    Also checks the classic MCQ dataset leak: is the CORRECT option
    systematically longer/shorter than the incorrect ones? If so, a
    model could learn 'pick the longest option' instead of real
    reasoning — inflating your MAP@3 in a way that won't generalize.
    """
    print("\n" + "=" * 60)
    print("TEXT LENGTH ANALYSIS")
    print("=" * 60)

    prompt_lens = train_df["prompt"].astype(str).str.split().str.len()
    print("\nPrompt word count:")
    print(prompt_lens.describe().round(1))

    correct_lens, incorrect_lens = [], []
    for _, row in train_df.iterrows():
        for opt in OPTIONS:
            length = len(str(row[opt]).split())
            if opt == row["answer"]:
                correct_lens.append(length)
            else:
                incorrect_lens.append(length)

    correct_avg = sum(correct_lens) / len(correct_lens)
    incorrect_avg = sum(incorrect_lens) / len(incorrect_lens)
    print(f"\nAvg word count — correct options:   {correct_avg:.2f}")
    print(f"Avg word count — incorrect options: {incorrect_avg:.2f}")
    diff_pct = abs(correct_avg - incorrect_avg) / incorrect_avg * 100
    if diff_pct > 10:
        print(f"  ⚠ {diff_pct:.0f}% difference — length may leak the answer. "
              f"Report this; consider a length-controlled baseline to confirm.")
    else:
        print(f"  ✓ Only {diff_pct:.0f}% difference — length doesn't look like a strong shortcut.")

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].hist(prompt_lens, bins=30, color="#55A868")
    axes[0].set_title("Prompt Word Count Distribution")
    axes[0].set_xlabel("Words")

    axes[1].hist([correct_lens, incorrect_lens], bins=20, label=["Correct", "Incorrect"],
                 color=["#4C72B0", "#C44E52"])
    axes[1].set_title("Option Word Count: Correct vs Incorrect")
    axes[1].set_xlabel("Words")
    axes[1].legend()

    fig.tight_layout()
  
    plt.close(fig)


def vocabulary_stats(train_df: pd.DataFrame, test_df: pd.DataFrame):
    print("\n" + "=" * 60)
    print("VOCABULARY")
    print("=" * 60)
    all_text = pd.concat([
        train_df["prompt"], test_df["prompt"],
        *[train_df[o] for o in OPTIONS], *[test_df[o] for o in OPTIONS],
    ]).astype(str)
    vocab = set()
    for text in all_text:
        vocab.update(text.lower().split())
    print(f"Approximate unique vocabulary size (whitespace split): {len(vocab)}")
    print("(This tells you roughly how large a TF-IDF vocabulary you're working with,"
          " and whether max_features in the baseline needs adjusting.)")



train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
basic_overview(train_df, test_df)
answer_distribution(train_df)
text_length_analysis(train_df)
vocabulary_stats(train_df, test_df)





BASIC OVERVIEW
Train shape: (2000, 8)   Test shape: (500, 7)

Train columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
Test columns:  ['id', 'prompt', 'A', 'B', 'C', 'D', 'E']

--- Missing values (train) ---
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

--- Missing values (test) ---
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
dtype: int64

Duplicate prompts in train: 242

ANSWER DISTRIBUTION (checking for positional bias)
  A:  369  (18.4%)
  B:  490  (24.5%)
  C:  459  (23.0%)
  D:  358  (17.9%)
  E:  324  (16.2%)

  ✓ Roughly uniform (max deviation 4.5pp) — no obvious positional bias.

TEXT LENGTH ANALYSIS

Prompt word count:
count    2000.0
mean       18.1
std         6.8
min         3.0
25%        14.0
50%        17.0
75%        22.0
max        51.0
Name: prompt, dtype: float64

Avg word count — correct options:   28.66
Avg word count — incorrect options: 25.68
  ⚠ 12% 